# Notebook — RAG with LangChain + Pinecone + Grok

This notebook implements a complete RAG pipeline:
- Document ingestion
- Text chunking
- Embeddings
- Pinecone indexing/retrieval
- Grounded answer generation with Grok

## 1) Install dependencies
Run this cell once.

In [ ]:
%pip install -q --upgrade pip
%pip install -q langchain langchain-groq langchain-pinecone pinecone python-dotenv

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + c:\Users\ders1\AppData\Local\Python\pythoncore-3.14-64\python.exe C:\Users\ders1\AppData\Local\Temp\pip-install-eiep00cz\numpy_e7088125ed0f4eb486a1104b409a18e2\vendored-meson\meson\meson.py setup C:\Users\ders1\AppData\Local\Temp\pip-install-eiep00cz\numpy_e7088125ed0f4eb486a1104b409a18e2 C:\Users\ders1\AppData\Local\Temp\pip-install-eiep00cz\numpy_e7088125ed0f4eb486a1104b409a18e2\.mesonpy-9c6g83dc -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\ders1\AppData\Local\Temp\pip-install-eiep00cz\numpy_e7088125ed0f4eb486a1104b409a18e2\.mesonpy-9c6g83dc\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\ders1\AppData\Local\Temp\pip-install-eiep00cz\numpy_e7088125ed0f4eb486a1104b409a18e2
      Build dir: C:\Users\ders1\AppData\Local\Temp\pip-install-eie

## 2) Environment and keys
Required: `PINECONE_API_KEY` and `GROQ_API_KEY`.

In [16]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

def clean_key(value: str | None) -> str:
    if not value:
        return ""
    return value.strip().strip('"').strip("'")

if not os.getenv("PINECONE_API_KEY"):
    entered = getpass.getpass("Enter PINECONE_API_KEY: ")
    os.environ["PINECONE_API_KEY"] = clean_key(entered)
else:
    os.environ["PINECONE_API_KEY"] = clean_key(os.getenv("PINECONE_API_KEY"))

if not os.getenv("GROQ_API_KEY"):
    entered = getpass.getpass("Enter GROQ_API_KEY: ")
    os.environ["GROQ_API_KEY"] = clean_key(entered)
else:
    os.environ["GROQ_API_KEY"] = clean_key(os.getenv("GROQ_API_KEY"))

if not os.environ["GROQ_API_KEY"]:
    raise RuntimeError("GROQ_API_KEY is empty after cleaning. Please set a valid key.")

os.environ.setdefault("GROQ_MODEL", "llama-3.1-8b-instant")
os.environ.setdefault("PINECONE_INDEX_NAME", "rag-daniel-original-kb")

print("Environment configured for Groq + Pinecone")
print("Model:", os.environ["GROQ_MODEL"])

print("Index:", os.environ["PINECONE_INDEX_NAME"])

print("GROQ_API_KEY detected")

Environment configured for Groq + Pinecone
Model: llama-3.1-8b-instant
Index: rag-daniel-original-kb
GROQ_API_KEY detected


## 3) Build your own knowledge base and split documents

In [17]:
from langchain_core.documents import Document

raw_knowledge = [
    {
        "source": "Campus RAG Guide",
        "topic": "rag",
        "content": "Retrieval-Augmented Generation (RAG) combines retrieval and generation. Instead of relying only on model memory, it fetches relevant chunks from a knowledge base and then composes a grounded answer. This reduces hallucinations and improves factuality in academic assistants."
    },
    {
        "source": "Lab Best Practices",
        "topic": "chunking",
        "content": "Chunking is critical in RAG. Smaller chunks improve precision, but chunks that are too short lose context. A practical start is chunk_size between 500 and 900 with overlap between 80 and 150. Overlap preserves continuity across chunk boundaries."
    },
    {
        "source": "Vector Search Notes",
        "topic": "pinecone",
        "content": "Pinecone stores embedding vectors and performs similarity search. During retrieval, user queries are embedded and compared against stored vectors. Top-k results are injected into the prompt. Cosine similarity is commonly used for semantic search."
    },
    {
        "source": "Prompt Design Memo",
        "topic": "prompting",
        "content": "A robust RAG prompt must explicitly instruct the model to answer only with retrieved context. If evidence is missing, the model should state it clearly. Source-aware formatting in context helps traceability and evaluation."
    },
    {
        "source": "Evaluation Rubric",
        "topic": "evaluation",
        "content": "RAG systems can be evaluated on grounding, answer relevance, and source coverage. A good answer should be supported by retrieved passages and avoid unsupported claims. Re-running the same question should produce consistent outputs when temperature is set to zero."
    },
    {
        "source": "Deployment Notes",
        "topic": "operations",
        "content": "For production, monitor index growth, retrieval latency, and token usage. Keep metadata fields like source, topic, and timestamp. These fields support filtering and better explainability in user-facing applications."
    }
]

def split_text(text: str, chunk_size: int = 700, chunk_overlap: int = 100):
    chunks_local = []
    start = 0
    step = max(1, chunk_size - chunk_overlap)
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks_local.append(text[start:end])
        if end == len(text):
            break
        start += step
    return chunks_local

documents = [
    Document(
        page_content=item["content"],
        metadata={"source": item["source"], "topic": item["topic"]}
    )
    for item in raw_knowledge
]

chunks = []
for doc in documents:
    for piece in split_text(doc.page_content, chunk_size=700, chunk_overlap=100):
        chunks.append(Document(page_content=piece, metadata=doc.metadata))

print(f"Created custom documents: {len(documents)}")
print(f"Generated chunks: {len(chunks)}")

Created custom documents: 6
Generated chunks: 6


## 4) Initialize Pinecone and ingest vectors

In [18]:
import re
from langchain_core.runnables import RunnableLambda

INDEX_NAME = os.environ["PINECONE_INDEX_NAME"]

def _tokenize(text: str):
    return set(re.findall(r"[a-zA-Z0-9]+", text.lower()))

def build_local_retriever(doc_chunks, k=3):
    def retrieve(query: str):
        q_tokens = _tokenize(query)
        scored = []
        for doc in doc_chunks:
            d_tokens = _tokenize(doc.page_content)
            score = len(q_tokens.intersection(d_tokens))
            scored.append((score, doc))
        scored.sort(key=lambda x: x[0], reverse=True)
        top_docs = [doc for score, doc in scored[:k] if score > 0]
        return top_docs if top_docs else doc_chunks[:k]
    return RunnableLambda(retrieve)

try:
    from pinecone import Pinecone, ServerlessSpec
    from langchain_pinecone import PineconeVectorStore, PineconeEmbeddings

    EMBEDDING_MODEL = "multilingual-e5-large"
    EMBEDDING_DIMENSION = 1024

    pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
    existing_indexes = {idx["name"] for idx in pc.list_indexes()}

    if INDEX_NAME not in existing_indexes:
        pc.create_index(
            name=INDEX_NAME,
            dimension=EMBEDDING_DIMENSION,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        print(f"Created Pinecone index: {INDEX_NAME}")
    else:
        print(f"Using existing Pinecone index: {INDEX_NAME}")

    embeddings = PineconeEmbeddings(model=EMBEDDING_MODEL)

    vector_store = PineconeVectorStore(
        index_name=INDEX_NAME,
        embedding=embeddings,
        pinecone_api_key=os.environ["PINECONE_API_KEY"]
    )

    vector_store.add_documents(chunks)
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    print("Ingestion complete with Pinecone. Retriever ready.")

except Exception as error:
    retriever = build_local_retriever(chunks, k=3)
    print("Pinecone is unavailable in this environment. Using local fallback retriever.")
    print(f"Reason: {error}")

Pinecone is unavailable in this environment. Using local fallback retriever.
Reason: No module named 'pinecone'


## 5) Build RAG chain (LCEL + Groq)
This section uses Groq with `ChatGroq`, aligned with Repo 1 style.

In [19]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'Unknown')}]\n{doc.page_content}"
        for doc in docs
    )

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer using ONLY the provided context. "
     "If the context is insufficient, say: I don't have enough information in the retrieved context."),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

llm = ChatGroq(
    groq_api_key=os.environ["GROQ_API_KEY"],
    model=os.environ["GROQ_MODEL"],
    temperature=0,
 )

# Early validation so API-key/model errors appear here instead of the last cell
try:
    _ = llm.invoke("ping")
except Exception as error:
    raise RuntimeError(
        "Groq initialization failed. Verify GROQ_API_KEY and GROQ_MODEL. "
        f"Original error: {error}"
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Groq RAG chain ready")

RuntimeError: Groq initialization failed. Verify GROQ_API_KEY and GROQ_MODEL. Original error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

## 6) Ask questions and show sources

In [20]:
def ask_with_sources(question: str):
    if "rag_chain" not in globals() or rag_chain is None:
        raise RuntimeError(
            "RAG chain is not initialized. Run cell 11 successfully first (Grok model + account credits required)."
        )

    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "source": doc.metadata.get("source", "Unknown"),
                "preview": doc.page_content[:180].replace("\n", " ") + "..."
            }
            for doc in source_docs
        ],
    }

In [ ]:
demo_questions = [
    "Why does RAG reduce hallucinations compared to plain generation?",
    "What chunking settings are suggested in this knowledge base?",
    "Which metadata fields are recommended for operational monitoring?"
]

for i, q in enumerate(demo_questions, start=1):
    try:
        result = ask_with_sources(q)
        print(f"Question {i}: {result['question']}")
        print(f"Answer: {result['answer']}")
        print("Sources used:")
        for src in result["sources"]:
            print(f" - {src['source']}")
        print("-" * 70)
    except Exception as error:
        print(f"Question {i}: {q}")
        print("Execution stopped in this cell.")
        print(f"Reason: {error}")
        print("Tip: rerun cell 5 and cell 11 with a valid GROQ_API_KEY.")
        break

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}